# Setup - Install Libraries

In [ ]:
# Run the following commands once, in order to install libraries - DO NOT Uncomment this line.

# Uncomment below lines

# !pip3 install --upgrade pip
# !pip install google-cloud-bigquery
# !pip install pandas-gbq -U
# !pip install db-dtypes
# !pip install packaging --upgrade

# Import libraries

In [1]:
# Import libraries
from google.cloud import bigquery
import pandas as pd
from pandas_gbq import to_gbq
import os

print('Libraries imported successfully')

Libraries imported successfully


In [2]:
# Set the environment variable for Google Cloud credentials
# Place the path in which the .json file is located.

# Example (if .json is located in the same directory with the notebook)
# os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "at-arch-416714-6f9900ec7.json"

# -- YOUR CODE GOES BELOW THIS LINE
os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "C:/Users/joank/Documents/Jupyter Notebook/capstone-project-495705-d8a776afa55d.json" # Edit path
# -- YOUR CODE GOES ABOVE THIS LINE

In [3]:
# Set your Google Cloud project ID and BigQuery dataset details

# -- YOUR CODE GOES BELOW THIS

project_id = 'capstone-project-495705' # Edit with your project id
dataset_id = 'staging_db' # Modify the necessary schema name: staging_db, reporting_db etc.
table_id = 'stg_customer' # Modify the necessary table name: stg_customer, stg_city etc.

# -- YOUR CODE GOES ABOVE THIS LINE

# SQL Query

In [7]:
# Create a BigQuery client
client = bigquery.Client(project=project_id)

# -- YOUR CODE GOES BELOW THIS LINE

# Define your SQL query here
query = """
with base as (
  select *
  from `capstone-project-495705.pagila_productionpublic.customer` --Your table path
  )

  , final as (
    select
        customer_id
        , store_id as customer_store_id
        , first_name as customer_first_name
        , last_name as customer_last_name
        , email as customer_email
        , address_id as customer_address_id
        , activebool as is_active_customer_bool
        , create_date as customer_create_date
        , last_update as customer_last_update
        , active as is_active_customer
   FROM base
  )

  select * from final
"""

# -- YOUR CODE GOES ABOVE THIS LINE

# Execute the query and store the result in a dataframe
df = client.query(query).to_dataframe()

# Explore some records
df.head()

C:\Users\joank\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\google\cloud\bigquery\table.py:2086: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


,customer_id,customer_store_id,customer_first_name,customer_last_name,customer_email,customer_address_id,is_active_customer_bool,customer_create_date,customer_last_update,is_active_customer
0,1,1,MARY,SMITH,MARY.SMITH@sakilacustomer.org,5,True,2022-02-14,2022-02-15 09:57:20+00:00,1
1,2,1,PATRICIA,JOHNSON,PATRICIA.JOHNSON@sakilacustomer.org,6,True,2022-02-14,2022-02-15 09:57:20+00:00,1
2,5,1,ELIZABETH,BROWN,ELIZABETH.BROWN@sakilacustomer.org,9,True,2022-02-14,2022-02-15 09:57:20+00:00,1
3,3,1,LINDA,WILLIAMS,LINDA.WILLIAMS@sakilacustomer.org,7,True,2022-02-14,2022-02-15 09:57:20+00:00,1
4,7,1,MARIA,MILLER,MARIA.MILLER@sakilacustomer.org,11,True,2022-02-14,2022-02-15 09:57:20+00:00,1


# Write to BigQuery

In [8]:
# Define the full table ID
full_table_id = f"{project_id}.{dataset_id}.{table_id}"

# -- YOUR CODE GOES BELOW THIS LINE
# Define table schema based on the project description

schema = [
    bigquery.SchemaField('customer_id', 'INTEGER'),
    bigquery.SchemaField('customer_store_id', 'INTEGER'),
    bigquery.SchemaField('customer_first_name', 'STRING'),
    bigquery.SchemaField('customer_last_name', 'STRING'),
    bigquery.SchemaField('customer_email', 'STRING'),
    bigquery.SchemaField('customer_address_id', 'INTEGER'),
    bigquery.SchemaField('is_active_customer_bool', 'BOOLEAN'),
    bigquery.SchemaField('customer_create_date', 'DATE'),
    bigquery.SchemaField('customer_last_update', 'DATETIME'),
    bigquery.SchemaField('is_active_customer', 'INTEGER'),
    ]

# -- YOUR CODE GOES ABOVE THIS LINE

In [9]:
# Create a BigQuery client
client = bigquery.Client(project=project_id)

# Check if the table exists
def table_exists(client, full_table_id):
    try:
        client.get_table(full_table_id)
        return True
    except Exception:
        return False

# Write the dataframe to the table (overwrite if it exists, create if it doesn't)
if table_exists(client, full_table_id):
    # If the table exists, overwrite it
    destination_table = f"{dataset_id}.{table_id}"
    # Write the dataframe to the table (overwrite if it exists)
    to_gbq(df, destination_table, project_id=project_id, if_exists='replace')
    print(f"Table {full_table_id} exists. Overwritten.")
else:
    # If the table does not exist, create it
    job_config = bigquery.LoadJobConfig(schema=schema)
    job = client.load_table_from_dataframe(df, full_table_id, job_config=job_config)
    job.result()  # Wait for the job to complete
    print(f"Table {full_table_id} did not exist. Created and data loaded.")

Table capstone-project-495705.staging_db.stg_customer exists. Overwritten.


In [ ]:
# Below line converts your i.pynb file to .py python executable file. Modify the input and output names based
# on the table you are processing.
# Example:
# ! jupyter nbconvert stg_customer.ipynb --to python

# -- YOUR CODE GOES BELOW THIS LINE

!python3 -m jupyter nbconvert stg_actor.ipynb --to python

# -- YOUR CODE GOES ABOVE THIS LINE

In [ ]:
!python3 -m pip install nbconvert

In [ ]:
!python3 -m pip install nbconvert -U